# LLMs Implementation for Tabular Classifications

На основе статьи: https://www.nb-data.com/p/llms-implementation-for-tabular-classifications

# Данные

Для примера используем данные прогноза оттока клиентов в телекоме с Каггла https://www.kaggle.com/datasets/barun2104/telecom-churn. 

Будем решать задачу бинарной классификации. Целевой колонкой (меткой) в данных является "Churn".

In [1]:
import pandas as pd

df = pd.read_csv('../data/telecom_churn.csv')
df.head()

,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
0,0,128,1,1,2.7,1,265.1,110,89.0,9.87,10.0
1,0,107,1,1,3.7,1,161.6,123,82.0,9.78,13.7
2,0,137,1,0,0.0,0,243.4,114,52.0,6.06,12.2
3,0,84,0,0,0.0,2,299.4,71,57.0,3.10,6.6
4,0,75,0,0,0.0,3,166.7,113,41.0,7.42,10.1


# Логистическая регрессия

Используем как базовое решение для сравнения.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [3]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Churn', axis =1), 
                                                    df['Churn'], 
                                                    test_size=.2, 
                                                    random_state = 42)

model = LogisticRegression(solver='liblinear')

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict_proba(X_test)[:, 1]

In [4]:
from sklearn.metrics import roc_auc_score
print('ROC AUC: ', roc_auc_score(y_test, y_pred)) 

ROC AUC:  0.8237588776545499


# LLM для классификации

### Подготовка данных

Переведем строки таблицы с признаками в тексты.

In [5]:
def concatenate_text(x):
    if x['ContractRenewal'] == 1:
      cr = 'have renew the contract'
    else:
      cr = 'never renew the contract'

    if x['DataPlan'] == 1:
      dp = 'have data plan'
    else:
      dp = "doesn't have data plan"

    full_text = (
        f"This customer account is {x['AccountWeeks']} weeks old, ",
        f"{cr}, ",
        f"{dp}, ",
        f"with {x['DataUsage']} GB of Monthly Data Usage, ",
        f"{x['CustServCalls']} times of Customer Service Calls, " ,
        f"{x['DayMins']} minutes total usage average monthly, ",
        f"{x['DayCalls']} times in average of daytime calls, "
        f"{x['MonthlyCharge']} monthly bill average, "
        f"with the largest overage Fee in the last 12 month is {x['OverageFee']}, "
        f"and {x['RoamMins']} minutes in average for roaming"

    )
    return ''.join(full_text)


X_train['label'] = y_train
X_test['label'] = y_test

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

In [6]:
X_train['text'].iloc[0]

"This customer account is 243.0 weeks old, have renew the contract, doesn't have data plan, with 0.0 GB of Monthly Data Usage, 2.0 times of Customer Service Calls, 95.5 minutes total usage average monthly, 92.0 times in average of daytime calls, 31.0 monthly bill average, with the largest overage Fee in the last 12 month is 8.19, and 6.6 minutes in average for roaming"

In [7]:
X_train['label'].iloc[0]

0

In [8]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
import evaluate

# Define label mappings
id2label = {0: "NOT-CHURN", 1: "CHURN"}
label2id = {"NOT-CHURN": 0, "CHURN": 1}

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Adjust based on the structure of your dataset
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

Map:   0%|          | 0/2666 [00:00<?, ? examples/s]

Map:   0%|          | 0/667 [00:00<?, ? examples/s]

Map:   0%|          | 0/2666 [00:00<?, ? examples/s]

Map:   0%|          | 0/667 [00:00<?, ? examples/s]

In [9]:
tokenized_train_dataset[0].keys()

dict_keys(['AccountWeeks', 'ContractRenewal', 'DataPlan', 'DataUsage', 'CustServCalls', 'DayMins', 'DayCalls', 'MonthlyCharge', 'OverageFee', 'RoamMins', 'label', 'text', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])

In [10]:
tokenized_train_dataset[0]['text']

"This customer account is 243.0 weeks old, have renew the contract, doesn't have data plan, with 0.0 GB of Monthly Data Usage, 2.0 times of Customer Service Calls, 95.5 minutes total usage average monthly, 92.0 times in average of daytime calls, 31.0 monthly bill average, with the largest overage Fee in the last 12 month is 8.19, and 6.6 minutes in average for roaming"

In [11]:
tokenized_train_dataset[0]['label']

0

### Дообучение модели

In [12]:
# Define the model with label mappings
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    learning_rate = 0.001,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch"
)

# Evaluation metric
auc = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    return auc.compute(prediction_scores=predictions, references=labels)

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Roc Auc
1,0.472300,0.447522,0.602727
2,0.571400,0.491342,0.642917
3,0.405700,0.432000,0.337237
4,0.414500,0.476025,0.393223
5,0.297200,0.427998,0.646188


TrainOutput(global_step=1670, training_loss=0.4450072478391453, metrics={'train_runtime': 2463.1909, 'train_samples_per_second': 5.412, 'train_steps_per_second': 0.678, 'total_flos': 876817591987200.0, 'train_loss': 0.4450072478391453, 'epoch': 5.0})

In [13]:
# Evaluate the model
results = trainer.evaluate()
print(results)

{'eval_loss': 0.4279976785182953, 'eval_roc_auc': 0.6461882937410349, 'eval_runtime': 21.58, 'eval_samples_per_second': 30.908, 'eval_steps_per_second': 3.893, 'epoch': 5.0}


### Примеры прогнозов

In [22]:
import torch

# Ensure the model is in evaluation mode and moved to CPU
model.eval()
model.to('cpu')

# Choose a specific instance to explain
idx = 2  # Index of the sample in your dataset
text_instance = X_test.iloc[idx]['text']

In [23]:
text_instance

"This customer account is 98.0 weeks old, have renew the contract, doesn't have data plan, with 0.0 GB of Monthly Data Usage, 4.0 times of Customer Service Calls, 0.0 minutes total usage average monthly, 0.0 times in average of daytime calls, 14.0 monthly bill average, with the largest overage Fee in the last 12 month is 7.98, and 6.8 minutes in average for roaming"

In [24]:
inputs = tokenizer(text_instance, return_tensors="pt", padding=True, truncation=True, max_length=128)

In [25]:
inputs

{'input_ids': tensor([[  101,  2023,  8013,  4070,  2003,  5818,  1012,  1014,  3134,  2214,
          1010,  2031, 20687,  1996,  3206,  1010,  2987,  1005,  1056,  2031,
          2951,  2933,  1010,  2007,  1014,  1012,  1014, 16351,  1997,  7058,
          2951,  8192,  1010,  1018,  1012,  1014,  2335,  1997,  8013,  2326,
          4455,  1010,  1014,  1012,  1014,  2781,  2561,  8192,  2779,  7058,
          1010,  1014,  1012,  1014,  2335,  1999,  2779,  1997, 12217,  4455,
          1010,  2403,  1012,  1014,  7058,  3021,  2779,  1010,  2007,  1996,
          2922,  2058,  4270,  7408,  1999,  1996,  2197,  2260,  3204,  2003,
          1021,  1012,  5818,  1010,  1998,  1020,  1012,  1022,  2781,  1999,
          2779,  2005, 24430,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [26]:
with torch.no_grad():
    logits = model(**inputs).logits

In [27]:
logits.numpy()

array([[ 0.8878014, -1.0507478]], dtype=float32)

In [28]:
torch.softmax(logits, dim=1).numpy()

array([[0.8741927 , 0.12580734]], dtype=float32)